In [ ]:
# XGBoost model example
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    average_precision_score,
    confusion_matrix,
    classification_report,
)
from xgboost import XGBClassifier
import plotly.graph_objects as go
import sys
sys.path.append('..')
from funs import *

# Example: load your data (replace with your actual data loading)
# data = pd.read_csv('your_data.csv')
# X = data.drop('target_column', axis=1)
# y = data['target_column']

# For demonstration, here is a placeholder for X and y:
# X, y = ...

# Split data (replace X and y with your actual variables)
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Initialize and fit XGBoost model
# model = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
# model.fit(X_train, y_train)

# Predict and evaluate
# y_pred = model.predict(X_test)
# print(classification_report(y_test, y_pred))
# print('Accuracy:', accuracy_score(y_test, y_pred))
# print('Confusion Matrix:\n', confusion_matrix(y_test, y_pred))

# Remove the comments and replace the placeholders with your actual data to use this template.

In [ ]:
data = get_df()
data

In [ ]:
categorical = [
    'Daytime/evening attendance', 
    'Debtor',
    'Tuition fees up to date',
    'Gender',
    'Scholarship holder'
]
numerical = []
target = ['Dropout']

In [ ]:
TEST_SIZE = 0.2

X = data[categorical + numerical]
y = (data['Dropout']).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=42, stratify=y
)

In [ ]:
RANDOM_STATE = 42

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical),
        ('num', StandardScaler(), numerical)
    ]
)

model = XGBClassifier(use_label_encoder=False, eval_metric='logloss')

pipe = Pipeline([
    ("preprocessor", preprocessor), 
    ("xgb", model)
])

pipe.fit(X_train, y_train)

In [ ]:
predictions = pipe.predict(X_test)
probabilities = pipe.predict_proba(X_test)

In [ ]:
def plot_roc_curves(y_true, scores_dict, out_html="roc_curve.html"):
    fig = go.Figure()
    for name, y_score in scores_dict.items():
        fpr, tpr, _ = roc_curve(y_true, y_score)
        auc = roc_auc_score(y_true, y_score)
        fig.add_trace(go.Scatter(x=fpr, y=tpr, mode="lines", name=f"{name} (AUC={auc:.3f})"))
    fig.add_trace(go.Scatter(x=[0,1], y=[0,1], mode="lines",
                             name="Chance", line=dict(dash="dash")))
    fig.update_layout(
        title="ROC Curve Comparison",
        xaxis_title="False Positive Rate",
        yaxis_title="True Positive Rate",
        template="plotly_white",
        legend_title_text=None,
        width=900, height=550
    )
    fig.write_html(out_html, include_plotlyjs="cdn")

def plot_pr_curves(y_true, scores_dict, out_html="pr_curve.html"):
    fig = go.Figure()
    pos_rate = (y_true.sum() / len(y_true)) if len(y_true) else 0.0
    for name, y_score in scores_dict.items():
        precision, recall, _ = precision_recall_curve(y_true, y_score)
        ap = average_precision_score(y_true, y_score)
        fig.add_trace(go.Scatter(x=recall, y=precision, mode="lines",
                                 name=f"{name} (AP={ap:.3f})"))
    fig.add_trace(go.Scatter(x=[0,1], y=[pos_rate, pos_rate], mode="lines",
                             name=f"Baseline (pos rate={float(pos_rate):.3f})",
                             line=dict(dash="dash")))
    fig.update_layout(
        title="Precision?Recall Curve",
        xaxis_title="Recall",
        yaxis_title="Precision",
        template="plotly_white",
        legend_title_text=None,
        width=900, height=550
    )
    fig.write_html(out_html, include_plotlyjs="cdn")

plot_roc_curves(y_test, {"XGBoost": probabilities[:, 1]}, out_html="xgb_roc_curve.html")
plot_pr_curves(y_test, {"XGBoost": probabilities[:, 1]}, out_html="xgb_pr_curve.html")

In [ ]:
y_preds = pipe.predict(X_test)
print(classification_report(y_test, y_preds))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_preds))